In [1]:
import shutil

import torch
from accelerate import PartialState
from datasets import load_from_disk
import random
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    HfArgumentParser,
)

from trl import (
    ModelConfig,
    PPOConfig,
    PPOTrainer,
    ScriptArguments,
    get_kbit_device_map,
    get_peft_config,
    get_quantization_config,
)
from trl.trainer.utils import SIMPLE_CHAT_TEMPLATE
import re
from alive2 import compare_llvm_ir_with_alive2

import llvmlite.binding as llvm

In [2]:
LLVM_TAG_RE = re.compile(r"<llvm>(.*?)</llvm>", re.I | re.S)
def extract_llvm_block(reply: str) -> str:
    """
    Return the text inside the <llvm> … </llvm> tags.
    Raises ValueError if the tags are missing.
    """
    m = LLVM_TAG_RE.search(reply)
    if not m:
        raise ValueError("No <llvm> … </llvm> block found in model output.")
    return m.group(1).strip()

llvm.initialize()

def count_instructions(ir_text: str) -> int:
    mod = llvm.parse_assembly(ir_text)
    mod.verify()
    return sum(
        1 for fn in mod.functions if not fn.is_declaration
        for bb in fn.blocks
        for _ in bb.instructions
    )

In [3]:
script_args = ScriptArguments(
    dataset_name=None,
    dataset_test_split=None,
)

training_args = PPOConfig(
    total_episodes=10,
    num_ppo_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    
    learning_rate=3e-6,
    sft_model_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    reward_model_path=None,
    stop_token="eos",
    missing_eos_penalty=1.0,
    use_cpu=True,
    bf16=False,
    fp16=False,
    output_dir="models/minimal/ppo_tldr",

    num_sample_generations=1,
    response_length=16,  

    eval_strategy="steps",
    logging_steps=1,   
    #save_strategy="no",
    #eval_steps=100,

    report_to="wandb",
    
    vf_coef = 0,
    temperature=0.6
    
)

model_args = ModelConfig(
    model_name_or_path="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    torch_dtype = 'float16'
)

In [4]:
shutil.rmtree(training_args.output_dir, ignore_errors=True)

In [5]:
torch_dtype = (
        model_args.torch_dtype if model_args.torch_dtype in ["auto", None] else getattr(torch, model_args.torch_dtype)
    )
print(torch_dtype)

torch.float16


In [6]:
quantization_config = get_quantization_config(model_args)
quantization_config

In [7]:
model_kwargs = dict(
        revision=model_args.model_revision,
        attn_implementation=model_args.attn_implementation,
        torch_dtype=torch_dtype,
        device_map=get_kbit_device_map() if quantization_config is not None else None,
        quantization_config=quantization_config,
    )

model_kwargs

{'revision': 'main',
 'attn_implementation': None,
 'torch_dtype': torch.float16,
 'device_map': None,
 'quantization_config': None}

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
        model_args.model_name_or_path, padding_side="left", trust_remote_code=model_args.trust_remote_code
    )
tokenizer

LlamaTokenizerFast(name_or_path='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', vocab_size=151643, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>', 'pad_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	151643: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<｜User｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151645: AddedToken("<｜Assistant｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151646: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|EOT|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	151648: AddedToken("<think>", rstrip=False

In [9]:
if tokenizer.chat_template is None:
    tokenizer.chat_template = SIMPLE_CHAT_TEMPLATE

In [10]:
print(tokenizer.chat_template)

{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}{%- for message in messages %}{%- if message['role'] == 'system' %}{% set ns.system_prompt = message['content'] %}{%- endif %}{%- endfor %}{{bos_token}}{{ns.system_prompt}}{%- for message in messages %}{%- if message['role'] == 'user' %}{%- set ns.is_tool = false -%}{{'<｜User｜>' + message['content']}}{%- endif %}{%- if message['role'] == 'assistant' and message['content'] is none %}{%- set ns.is_tool = false -%}{%- for tool in message['tool_calls']%}{%- if not ns.is_first %}{{'<｜Assistant｜><｜tool▁calls▁begin｜><｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\n' + '```json' + '\n' + tool['function']['arguments'] + '\n' + '```' + '<｜tool▁call▁end｜>'}}{%- set ns.is_first = true -%}{%- else %}{{'\n' + '<｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['functio

In [11]:
class LLVMRewardModel(torch.nn.Module):
    base_model_prefix = "critic_backbone"
    def __init__(self, alive_tv_path: str, success_reward: float = 5.0, failure_penalty: float = -1.0, same_penalty: float = -2.0, inst_bonus: float = 1.0):
        super().__init__()
        self.alive_tv_path = alive_tv_path
        self.success_reward = success_reward
        self.failure_penalty = failure_penalty
        self.same_penalty = same_penalty
        self.inst_bonus = inst_bonus

    @property
    def critic_backbone(self):
        return self
    def named_children(self):
        return iter([])

    def forward(self, input_ids=None, attention_mask=None, non_optimized=None, llvm_instcombine=None, **kwargs):
        # Decode the full model reply to text
        reply = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        try:
            # Extract only the optimized IR block
            gen_ir = extract_llvm_block(reply)
        except ValueError:
            # No <llvm>…</llvm> found: treat as overall failure
            batch_size, seq_len = input_ids.shape
            return torch.full((batch_size, seq_len, 1), self.failure_penalty, device=input_ids.device)
            
        # Compare with non_optimized
        result = compare_llvm_ir_with_alive2(non_optimized[0], gen_ir, self.alive_tv_path)
        status = result['status']
        reward = 0.0
        if status == 'success':
            reward += self.success_reward
            # instruction count bonus
            if count_instructions(gen_ir) < count_instructions(non_optimized[0]):
                reward += self.inst_bonus
        else:
            reward += self.failure_penalty
        # penalty if same as non_optimized
        if gen_ir.strip() == non_optimized[0].strip():
            reward += self.same_penalty
        if gen_ir.strip() == llvm_instcombine[0].strip():
            reward += self.success_reward
        # return a constant reward tensor of shape (batch, seq_len, 1)
        batch_size, seq_len = input_ids.shape
        return torch.full((batch_size, seq_len, 1), reward, device=input_ids.device)((batch_size, seq_len, 1), reward, device=input_ids.device)

alive2_path = "/opt/homebrew/bin/alive-tv"
elementary_model = LLVMRewardModel(alive2_path)
value_model = elementary_model
reward_model = elementary_model

In [12]:
policy = AutoModelForCausalLM.from_pretrained(
    training_args.sft_model_path, trust_remote_code=model_args.trust_remote_code
)

In [13]:
peft_config = get_peft_config(model_args)
peft_config

In [14]:
if peft_config is None:
    ref_policy = AutoModelForCausalLM.from_pretrained(
        training_args.sft_model_path, trust_remote_code=model_args.trust_remote_code
    )
else:
    ref_policy = None

In [15]:
dataset = load_from_disk("processed_data_with_prompt")
train_dataset = dataset
eval_dataset = dataset.select(list(range(min(1000, len(dataset)))))

# # Perform a random train/test split: 90% train, 10% test
# split = dataset.train_test_split(test_size=0.1, seed=training_args.seed)
# train_dataset = split["train"]
# test_dataset = split["test"] if training_args.eval_strategy != "no" else None
# dataset = load_dataset(script_args.dataset_name, name=script_args.dataset_config)
# train_dataset = dataset[script_args.dataset_train_split]
# eval_dataset = dataset[script_args.dataset_test_split] if training_args.eval_strategy != "no" else None

In [16]:
def prepare_dataset(ds, tokenizer):
    def tokenize(element):
        encoding = tokenizer(
            element["prompt"],
            add_special_tokens=False,
            return_attention_mask=True,
            truncation=True,
        )
        return {
            "prompt": element["prompt"],       
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "non_optimized": element["non_optimized"],
            "llvm_instcombine": element["LLVM_instcombine"],
        }

    return ds.map(
        tokenize,
        remove_columns=ds.column_names,
        num_proc=training_args.dataset_num_proc,
    )

In [17]:
with PartialState().local_main_process_first():
    train_dataset = prepare_dataset(train_dataset, tokenizer)
    eval_dataset = prepare_dataset(eval_dataset, tokenizer)

Map:   0%|          | 0/34190 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [18]:
def custom_collator(features: list[dict]):
    # Build encodings list for padding
    encodings = [
        {"input_ids": f["input_ids"], "attention_mask": f["attention_mask"]}
        for f in features
    ]
    # Tokenizer.pad will pad both input_ids and attention_mask
    batch = tokenizer.pad(
        encodings,
        padding=True,
        return_tensors="pt",
    )
    # Attach metadata lists for reward computation
    batch["non_optimized"] = [f["non_optimized"] for f in features]
    batch["llvm_instcombine"] = [f["llvm_instcombine"] for f in features]
    return batch

In [19]:
ppo_trainer = PPOTrainer(
    args=training_args,              
    model=policy,
    ref_model=ref_policy,
    reward_model=elementary_model,   
    value_model=elementary_model,     
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,     
    peft_config=peft_config,
    data_collator=custom_collator
)

In [20]:
train_dataset

Dataset({
    features: ['non_optimized', 'prompt', 'input_ids', 'attention_mask', 'llvm_instcombine'],
    num_rows: 34190
})

In [29]:
def compute_llvm_reward(raw_ir: str, instcombine_ir: str, reply: str) -> float:
    # 1) Extract LLVM block
    try:
        gen_block = extract_llvm_block(reply)
    except ValueError:
        return elementary_model.failure_penalty

    # 2) Compare with Alive2
    try:
        result = compare_llvm_ir_with_alive2(raw_ir, gen_block, alive2_path)
        status = result.get("status", "failure")
    except Exception:
        return elementary_model.failure_penalty

    # 3) Base reward
    if status != "success":
        return elementary_model.failure_penalty
    r = elementary_model.success_reward

    # 4) Instruction count bonus
    orig_cnt = count_instructions(raw_ir)
    new_cnt = count_instructions(gen_block)
    if orig_cnt and new_cnt and new_cnt < orig_cnt:
        r += elementary_model.inst_bonus

    # 5) Penalty if identical to non-optimized
    if gen_block.strip() == raw_ir.strip():
        r += elementary_model.same_penalty

    # 6) Bonus if matches instcombine
    if gen_block.strip() == instcombine_ir.strip():
        r += elementary_model.success_reward

    return r

In [30]:
for episode in range(training_args.total_episodes):
    ex = train_dataset[episode]
    prompt         = ex["prompt"]
    raw_ir         = ex["non_optimized"]
    instcombine_ir = ex["llvm_instcombine"]

    inputs = tokenizer(prompt, return_tensors="pt").to(policy.device)
    with torch.no_grad():
        gen_ids = policy.generate(
            **inputs,
            max_new_tokens=training_args.response_length,
            eos_token_id=tokenizer.eos_token_id,
        )
    reply = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
    print(reply)

    # compute scalar reward (handles missing tags internally)
    reward = compute_llvm_reward(raw_ir, instcombine_ir, reply)

    ppo_trainer.step([prompt], [reply], [reward])

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<｜User｜>You are an LLVM optimisation assistant. Given the non‑optimised LLVM IR below, start your response with '<think>' on a new line, reason step by step inside <think>…</think> tags, and then output the optimised IR inside <llvm>…</llvm> tags.

; ModuleID = '<stdin>'
source_filename = "-"
target datalayout = "e-m:e-i8:8:32-i16:16:32-i64:64-i128:128-n32:64-S128"
target triple = "aarch64-unknown-linux-gnu"

; Function Attrs: noinline nounwind uwtable
define dso_local void @e7(i64 noundef %0) #0 {
  %2 = alloca i64, align 8
  store i64 %0, i64* %2, align 8
  %3 = load i64, i64* %2, align 8
  call void @longjmp(i64 noundef %3, i32 noundef 1)
  ret void
}

declare void @longjmp(i64 noundef, i32 noundef) #1

attributes #0 = { noinline nounwind uwtable "frame-pointer"="non-leaf" "min-legal-vector-width"="0" "no-trapping-math"="true" "stack-protector-buffer-size"="8" "target-cpu"="generic" "target-features"="+neon,+outline-atomics" }
attributes #1 = { "frame-pointer"="non-leaf" "no-trappin

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


AttributeError: 'PPOTrainer' object has no attribute 'step'

In [20]:
#trainer.generate_completions(sampling=True)